# BI Final Project — Online Retail Customer Segmentation

**Course:** Business Intelligence  
**Topic:** Customer segmentation for targeted marketing  
**Dataset:** Online Retail transactions  
**Model:** K-Means clustering with **k = 3**  

This notebook follows the five stages of the BI project:

1. **Frame & KPIs**
2. **Prepare data and EDA**
3. **Model & evaluate**
4. **Communicate results and recommendations**
5. **Ethics & limitations**

The goal is to create a useful analysis for a marketing team. We keep the code understandable, but still include enough EDA, evaluation, charts, and a dashboard to support a final presentation.

# STAGE 1 — Frame & KPIs

## Business question

How can an online retail company segment its customers based on purchasing behavior using **Recency, Frequency, and Monetary value (RFM)** to identify high-value, at-risk, and low-engagement groups for targeted marketing?

## Decision-maker

The decision-maker is the **Marketing Manager / E-commerce Manager**.

## Business decision

The marketing team will use the customer segments to decide which customers should receive:

- VIP loyalty campaigns
- Win-back or reactivation campaigns
- Regular promotional campaigns
- Cross-selling or bundle offers

## Main KPIs

| KPI | Meaning | Why it matters |
|---|---|---|
| Recency | Days since the last purchase | Helps identify active or inactive customers |
| Frequency | Number of purchases | Shows how often customers buy |
| Monetary | Total spending | Shows customer value |
| Average Order Value | Monetary / Frequency | Helps identify big-ticket customers |
| Customers by segment | Number of customers in each segment | Helps estimate campaign size |
| Revenue by segment | Total monetary value per segment | Helps prioritize marketing effort |

## Model type

This is an **unsupervised learning** problem because the dataset does not have a target variable like `Churn`, `HighValue`, or `CustomerType`.  
We use **K-Means clustering** to discover customer groups from their purchasing behavior.

# Setup

The notebook can run if the dataset is in the same folder as this notebook as either:

- `online+retail.zip`, or
- `Online Retail.xlsx`

The code uses pandas, scikit-learn, and matplotlib.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import base64
from pathlib import Path
from IPython.display import display, HTML

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

RANDOM_STATE = 42

# Simple visual settings for clearer charts
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Load the dataset

This cell loads the Excel file. If the Excel file is inside the ZIP file, the notebook extracts it automatically.

In [ ]:
zip_path = Path("online+retail.zip")
excel_path = Path("Online Retail.xlsx")

if excel_path.exists():
    data_file = excel_path
elif zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("data")
    data_file = Path("data") / "Online Retail.xlsx"
else:
    raise FileNotFoundError("Please upload online+retail.zip or Online Retail.xlsx to the same folder as this notebook.")

print("Using file:", data_file)

df = pd.read_excel(data_file)
display(df.head())

# STAGE 2 — Prepare data and EDA

Before modeling, we need to understand the original dataset and then clean it.  
EDA means **Exploratory Data Analysis**. It helps us see data quality problems, missing values, patterns, and possible business insights.

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
missing_table = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_values", ascending=False)

missing_table

In [ ]:
display(df.describe())

## Basic quality checks

We check possible data quality issues before cleaning:

- Missing `CustomerID`
- Negative or zero quantity
- Negative or zero unit price
- Cancelled invoices, usually marked with invoice numbers that start with `C`

In [ ]:
quality_checks = pd.DataFrame({
    "Check": [
        "Total rows",
        "Missing CustomerID",
        "Quantity <= 0",
        "UnitPrice <= 0",
        "Cancelled invoices starting with C"
    ],
    "Rows": [
        len(df),
        df["CustomerID"].isna().sum(),
        (df["Quantity"] <= 0).sum(),
        (df["UnitPrice"] <= 0).sum(),
        df["InvoiceNo"].astype(str).str.startswith("C").sum()
    ]
})

quality_checks

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(quality_checks["Check"], quality_checks["Rows"])
ax.set_title("Initial Data Quality Checks")
ax.set_xlabel("Number of rows")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Cleaning rules

We apply clear and simple cleaning rules:

1. Remove rows with missing `CustomerID` because customer segmentation needs a customer ID.
2. Remove cancelled invoices because the goal is to analyze real purchases.
3. Keep only positive quantities.
4. Keep only positive unit prices.
5. Create a new column called `Revenue = Quantity * UnitPrice`.

These rules make the customer-level RFM calculation more reliable.

In [ ]:
df_clean = df.copy()
rows_original = len(df_clean)

# Remove missing CustomerID
rows_before_customer = len(df_clean)
df_clean = df_clean.dropna(subset=["CustomerID"])
rows_after_customer = len(df_clean)

# Remove cancelled invoices
rows_before_cancelled = len(df_clean)
df_clean = df_clean[~df_clean["InvoiceNo"].astype(str).str.startswith("C")]
rows_after_cancelled = len(df_clean)

# Keep valid quantities and prices
rows_before_positive = len(df_clean)
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)]
rows_after_positive = len(df_clean)

# Create revenue
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

cleaning_summary = pd.DataFrame({
    "Step": [
        "Original dataset",
        "After removing missing CustomerID",
        "After removing cancelled invoices",
        "After keeping only positive Quantity and UnitPrice"
    ],
    "Rows": [
        rows_original,
        rows_after_customer,
        rows_after_cancelled,
        rows_after_positive
    ]
})

cleaning_summary["Rows_removed_from_previous_step"] = cleaning_summary["Rows"].shift(1) - cleaning_summary["Rows"]
cleaning_summary["Rows_removed_from_previous_step"] = cleaning_summary["Rows_removed_from_previous_step"].fillna(0).astype(int)

cleaning_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cleaning_summary["Step"], cleaning_summary["Rows"])
ax.set_title("Rows Remaining After Each Cleaning Step")
ax.set_ylabel("Rows")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## EDA after cleaning

Now we check the cleaned dataset to understand the business context.

In [ ]:
overall_kpis = pd.DataFrame({
    "Metric": [
        "Clean rows",
        "Unique customers",
        "Unique invoices",
        "Unique products",
        "Countries",
        "Total revenue",
        "Average invoice revenue",
        "First invoice date",
        "Last invoice date"
    ],
    "Value": [
        f"{len(df_clean):,}",
        f"{df_clean['CustomerID'].nunique():,}",
        f"{df_clean['InvoiceNo'].nunique():,}",
        f"{df_clean['StockCode'].nunique():,}",
        f"{df_clean['Country'].nunique():,}",
        f"£{df_clean['Revenue'].sum():,.2f}",
        f"£{df_clean.groupby('InvoiceNo')['Revenue'].sum().mean():,.2f}",
        str(df_clean['InvoiceDate'].min()),
        str(df_clean['InvoiceDate'].max())
    ]
})

overall_kpis

In [ ]:
country_revenue = df_clean.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(country_revenue.index[::-1], country_revenue.values[::-1])
ax.set_title("Top 10 Countries by Revenue")
ax.set_xlabel("Revenue")
plt.tight_layout()
plt.show()

In [ ]:
country_transactions = df_clean["Country"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(country_transactions.index[::-1], country_transactions.values[::-1])
ax.set_title("Top 10 Countries by Number of Transactions")
ax.set_xlabel("Transactions")
plt.tight_layout()
plt.show()

In [ ]:
monthly_revenue = df_clean.set_index("InvoiceDate").resample("M")["Revenue"].sum()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(monthly_revenue.index, monthly_revenue.values, marker="o")
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
plt.tight_layout()
plt.show()

In [ ]:
top_products = (
    df_clean.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_products.index[::-1], top_products.values[::-1])
ax.set_title("Top 10 Products by Revenue")
ax.set_xlabel("Revenue")
plt.tight_layout()
plt.show()

In [ ]:
invoice_revenue = df_clean.groupby("InvoiceNo")["Revenue"].sum()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(invoice_revenue, bins=60)
ax.set_title("Distribution of Invoice Revenue")
ax.set_xlabel("Invoice revenue")
ax.set_ylabel("Number of invoices")
ax.set_xlim(0, invoice_revenue.quantile(0.99))
plt.tight_layout()
plt.show()

## EDA findings

From the EDA, we learned that:

- The dataset has enough rows, customers, products, and countries for a BI project.
- Some rows must be removed because they are cancelled, missing customer IDs, or invalid purchases.
- Revenue is not equally distributed: some countries, products, and customers contribute much more than others.
- This supports the idea of segmentation, because not all customers should receive the same marketing action.

# Feature engineering — RFM table

We now convert the transaction-level data into a customer-level table.

RFM variables:

- **Recency:** days since the last purchase. Lower is better.
- **Frequency:** number of invoices/purchases. Higher is better.
- **Monetary:** total revenue from the customer. Higher is better.

We also add **Average Order Value** for interpretation.

In [ ]:
reference_date = df_clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

rfm["AvgOrderValue"] = rfm["Monetary"] / rfm["Frequency"]

rfm.head()

In [ ]:
rfm.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(rfm["Recency"], bins=40)
axes[0].set_title("Recency Distribution")
axes[0].set_xlabel("Days since last purchase")

axes[1].hist(rfm["Frequency"], bins=40)
axes[1].set_title("Frequency Distribution")
axes[1].set_xlabel("Number of purchases")
axes[1].set_xlim(0, rfm["Frequency"].quantile(0.99))

axes[2].hist(rfm["Monetary"], bins=40)
axes[2].set_title("Monetary Distribution")
axes[2].set_xlabel("Total spending")
axes[2].set_xlim(0, rfm["Monetary"].quantile(0.99))

plt.tight_layout()
plt.show()

## Preparing RFM for K-Means

K-Means uses distance. Because Recency, Frequency, and Monetary are on different scales, we scale them.

We also apply `log1p` before scaling. This keeps the analysis simple, but reduces the effect of very large values.

In [ ]:
rfm_model = rfm[["Recency", "Frequency", "Monetary"]].copy()
rfm_log = np.log1p(rfm_model)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

print("Data used by K-Means:", rfm_scaled.shape)

# STAGE 3 — Model and evaluation

We test different values of `k` to see how many clusters could work.

We use:

- **Inertia:** lower values mean clusters are more compact.
- **Silhouette score:** higher values mean clusters are better separated.

For the final project, we also consider **business interpretability**. The clusters must be easy to explain to a marketing manager.

In [ ]:
k_values = range(2, 9)
inertia_values = []
silhouette_values = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(rfm_scaled)
    inertia_values.append(model.inertia_)
    silhouette_values.append(silhouette_score(rfm_scaled, labels))

k_results = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertia_values,
    "silhouette": silhouette_values
})

k_results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(k_results["k"], k_results["inertia"], marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_results["k"], k_results["silhouette"], marker="o")
axes[1].set_title("Silhouette Score by k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")

plt.tight_layout()
plt.show()

## Final choice: k = 3

We choose **k = 3** because the project goal is to identify three marketing groups:

1. **High-value customers**
2. **At-risk / low-engagement customers**
3. **Regular customers with growth potential**

This value of `k` is simple enough to explain in a stakeholder presentation, and it is aligned with the business question. We still show the elbow and silhouette analysis to make the choice defensible.

In [ ]:
FINAL_K = 3

kmeans = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

rfm["Cluster"].value_counts().sort_index()

In [ ]:
cluster_summary = rfm.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Avg_Order_Value=("AvgOrderValue", "mean"),
    Total_Revenue=("Monetary", "sum")
).round(2)

cluster_summary["Revenue_Share_%"] = (cluster_summary["Total_Revenue"] / cluster_summary["Total_Revenue"].sum() * 100).round(2)

cluster_summary

## Naming the clusters

K-Means only gives cluster numbers. These numbers do not have business meaning by themselves.

We translate the clusters into marketing names using their average RFM values:

- The segment with the highest monetary value becomes **High-Value Customers**.
- The segment with the highest recency becomes **At-Risk / Low-Engagement Customers**.
- The remaining segment becomes **Regular / Growth Potential Customers**.

In [ ]:
high_value_cluster = cluster_summary["Avg_Monetary"].idxmax()
low_engagement_cluster = cluster_summary["Avg_Recency"].idxmax()

segment_names = {}
for cluster in cluster_summary.index:
    if cluster == high_value_cluster:
        segment_names[cluster] = "High-Value Customers"
    elif cluster == low_engagement_cluster:
        segment_names[cluster] = "At-Risk / Low-Engagement Customers"
    else:
        segment_names[cluster] = "Regular / Growth Potential Customers"

rfm["Segment"] = rfm["Cluster"].map(segment_names)
segment_names

In [ ]:
segment_summary = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Avg_Order_Value=("AvgOrderValue", "mean"),
    Total_Revenue=("Monetary", "sum")
).round(2)

segment_summary["Revenue_Share_%"] = (segment_summary["Total_Revenue"] / segment_summary["Total_Revenue"].sum() * 100).round(2)

segment_order = [
    "High-Value Customers",
    "Regular / Growth Potential Customers",
    "At-Risk / Low-Engagement Customers"
]
segment_summary = segment_summary.reindex(segment_order)

segment_summary

In [ ]:
recommendations = pd.DataFrame({
    "Segment": [
        "High-Value Customers",
        "Regular / Growth Potential Customers",
        "At-Risk / Low-Engagement Customers"
    ],
    "Business meaning": [
        "Best customers. They generate high revenue and are very important for the business.",
        "Customers with moderate behavior who could buy more with the right offer.",
        "Customers who have not purchased recently and may be inactive or close to being lost."
    ],
    "Recommended action": [
        "Give VIP benefits, loyalty rewards, early access, and personalized offers.",
        "Use cross-selling, product recommendations, bundles, or small discounts.",
        "Use low-cost win-back emails or reactivation campaigns. Avoid expensive campaigns first."
    ],
    "KPI to monitor": [
        "Retention and revenue from VIP customers",
        "Increase in frequency and average order value",
        "Return purchase rate after the win-back campaign"
    ]
})

recommendations

## PCA visualization

The model uses three RFM variables, so it is hard to see the clusters directly.  
PCA reduces the model data to two dimensions only for visualization. It is not an accuracy metric.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
rfm_pca = pca.fit_transform(rfm_scaled)

pca_df = pd.DataFrame({
    "PC1": rfm_pca[:, 0],
    "PC2": rfm_pca[:, 1],
    "Segment": rfm["Segment"]
})

fig, ax = plt.subplots(figsize=(8, 5))
for segment in segment_order:
    temp = pca_df[pca_df["Segment"] == segment]
    ax.scatter(temp["PC1"], temp["PC2"], label=segment, alpha=0.6, s=25)

ax.set_title("Customer Segments Visualized with PCA")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# STAGE 4 — Communicate results and recommendations

This stage turns the analysis into visuals and a dashboard for the stakeholder.

In [ ]:
def save_bar_chart(series, title, xlabel, filename):
    # Create and save a horizontal bar chart.
    fig, ax = plt.subplots(figsize=(8, 4.5))
    series = series.dropna()
    ax.barh(series.index[::-1], series.values[::-1])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    path = output_dir / filename
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

customers_chart = save_bar_chart(
    segment_summary["Customers"],
    "Number of Customers by Segment",
    "Customers",
    "customers_by_segment.png"
)

revenue_chart = save_bar_chart(
    segment_summary["Total_Revenue"],
    "Total Revenue by Segment",
    "Revenue",
    "revenue_by_segment.png"
)

recency_chart = save_bar_chart(
    segment_summary["Avg_Recency"],
    "Average Recency by Segment",
    "Days since last purchase",
    "recency_by_segment.png"
)

frequency_chart = save_bar_chart(
    segment_summary["Avg_Frequency"],
    "Average Frequency by Segment",
    "Average number of purchases",
    "frequency_by_segment.png"
)

monetary_chart = save_bar_chart(
    segment_summary["Avg_Monetary"],
    "Average Monetary Value by Segment",
    "Average monetary value",
    "monetary_by_segment.png"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(monthly_revenue.index, monthly_revenue.values, marker="o")
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
plt.tight_layout()
monthly_chart = output_dir / "monthly_revenue_trend.png"
plt.savefig(monthly_chart, dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Save data outputs for the final ZIP and presentation
rfm.to_csv(output_dir / "customer_segments_output.csv", index=False)
segment_summary.to_csv(output_dir / "segment_summary_output.csv")
recommendations.to_csv(output_dir / "marketing_recommendations.csv", index=False)
k_results.to_csv(output_dir / "k_evaluation_results.csv", index=False)

overall_kpis.to_csv(output_dir / "overall_kpis.csv", index=False)
cleaning_summary.to_csv(output_dir / "cleaning_summary.csv", index=False)

print("CSV and chart files saved in the outputs folder.")

## Create an HTML dashboard

This dashboard is self-contained: it embeds the charts into the HTML file.  
The file can be included in the final ZIP or used to take screenshots for the presentation.

In [ ]:
def image_to_base64(path):
    with open(path, "rb") as image_file:
        encoded = base64.b64encode(image_file.read()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"

chart_images = {
    "customers": image_to_base64(customers_chart),
    "revenue": image_to_base64(revenue_chart),
    "recency": image_to_base64(recency_chart),
    "frequency": image_to_base64(frequency_chart),
    "monetary": image_to_base64(monetary_chart),
    "monthly": image_to_base64(monthly_chart)
}

cards_html = ""
for _, row in overall_kpis.head(6).iterrows():
    cards_html += f'''
    <div class="card">
        <div class="card-title">{row["Metric"]}</div>
        <div class="card-value">{row["Value"]}</div>
    </div>
    '''

html = f'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Online Retail Customer Segmentation Dashboard</title>
<style>
    body {{
        margin: 0;
        font-family: Arial, sans-serif;
        background: #f4f6f8;
        color: #1f2937;
    }}
    header {{
        background: linear-gradient(135deg, #1f2937, #2563eb);
        color: white;
        padding: 32px 48px;
    }}
    header h1 {{
        margin: 0;
        font-size: 30px;
    }}
    header p {{
        margin-top: 8px;
        max-width: 900px;
        line-height: 1.5;
    }}
    main {{
        padding: 28px 48px;
    }}
    .grid {{
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));
        gap: 16px;
        margin-bottom: 26px;
    }}
    .card {{
        background: white;
        padding: 18px;
        border-radius: 14px;
        box-shadow: 0 4px 12px rgba(0,0,0,0.08);
    }}
    .card-title {{
        color: #6b7280;
        font-size: 13px;
        text-transform: uppercase;
        letter-spacing: 0.05em;
    }}
    .card-value {{
        font-size: 24px;
        font-weight: bold;
        margin-top: 8px;
        color: #111827;
    }}
    .section {{
        background: white;
        border-radius: 14px;
        padding: 22px;
        margin-bottom: 24px;
        box-shadow: 0 4px 12px rgba(0,0,0,0.08);
    }}
    h2 {{
        margin-top: 0;
        color: #111827;
    }}
    table {{
        width: 100%;
        border-collapse: collapse;
        margin-top: 12px;
        font-size: 14px;
    }}
    th {{
        background: #e5e7eb;
        text-align: left;
        padding: 10px;
    }}
    td {{
        border-bottom: 1px solid #e5e7eb;
        padding: 10px;
        vertical-align: top;
    }}
    .charts {{
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(420px, 1fr));
        gap: 18px;
    }}
    .chart-card {{
        background: #ffffff;
        border: 1px solid #e5e7eb;
        border-radius: 12px;
        padding: 12px;
    }}
    .chart-card img {{
        width: 100%;
        border-radius: 8px;
    }}
    .recommendation {{
        border-left: 5px solid #2563eb;
        background: #eff6ff;
        padding: 16px;
        border-radius: 10px;
        line-height: 1.5;
    }}
    footer {{
        padding: 20px 48px;
        color: #6b7280;
        font-size: 13px;
    }}
</style>
</head>
<body>
<header>
    <h1>Online Retail Customer Segmentation Dashboard</h1>
    <p>This dashboard summarizes customer segments created with K-Means clustering using RFM variables: Recency, Frequency, and Monetary value.</p>
</header>
<main>
    <div class="grid">
        {cards_html}
    </div>

    <div class="section">
        <h2>Segment Summary</h2>
        {segment_summary.to_html(classes="data-table")}
    </div>

    <div class="section">
        <h2>Marketing Recommendations</h2>
        {recommendations.to_html(index=False, classes="data-table")}
    </div>

    <div class="section">
        <h2>Dashboard Charts</h2>
        <div class="charts">
            <div class="chart-card"><img src="{chart_images['customers']}" alt="Customers by segment"></div>
            <div class="chart-card"><img src="{chart_images['revenue']}" alt="Revenue by segment"></div>
            <div class="chart-card"><img src="{chart_images['recency']}" alt="Recency by segment"></div>
            <div class="chart-card"><img src="{chart_images['frequency']}" alt="Frequency by segment"></div>
            <div class="chart-card"><img src="{chart_images['monetary']}" alt="Monetary by segment"></div>
            <div class="chart-card"><img src="{chart_images['monthly']}" alt="Monthly revenue trend"></div>
        </div>
    </div>

    <div class="section recommendation">
        <h2>Main Recommendation</h2>
        <p>The marketing team should not send the same campaign to every customer. High-value customers should receive loyalty benefits, regular customers should receive cross-selling and bundle offers, and at-risk customers should receive low-cost win-back campaigns.</p>
    </div>
</main>
<footer>
    Generated from the BI Final Project notebook. Results depend on the cleaned Online Retail dataset and the selected k = 3 segmentation.
</footer>
</body>
</html>
'''

html_path = Path("online_retail_segmentation_dashboard.html")
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html)

print("HTML dashboard saved as:", html_path)
display(HTML(f'<a href="{html_path}" target="_blank">Open HTML dashboard</a>'))

## Main business recommendation

The marketing team should use the segmentation to personalize campaigns:

1. **High-Value Customers:** protect them with VIP loyalty benefits and personalized offers.
2. **Regular / Growth Potential Customers:** increase their value with bundles, product recommendations, and cross-selling.
3. **At-Risk / Low-Engagement Customers:** use low-cost reactivation emails or discounts, because expensive campaigns may not be profitable.

This connects the model to a real business decision: **who receives which marketing action**.

# STAGE 5 — Ethics, bias, and limitations

## Privacy

The dataset uses `CustomerID`, but it does not include names, emails, phone numbers, or addresses. Even so, customer IDs still represent real customer behavior and should be treated carefully.

## Bias and fairness

The dataset may be concentrated in specific countries and one retail context. Because of this, the results may not generalize to all markets or all customer types.

## Limitations

- This is a segmentation model, not a prediction model.
- The model does not prove why customers behave in a certain way.
- The data is historical, so customer behavior may have changed.
- K-Means depends on the selected variables and the value of `k`.
- The cluster names are business interpretations, not automatic truths.
- Returns and cancellations were removed, so the model does not analyze return behavior.
- A/B testing would be needed to prove that the recommended campaigns actually improve revenue or retention.

## Responsible use

The segments should support marketing decisions, but they should not be used to unfairly exclude customers. The company should monitor campaign results and update the segmentation over time.

# Final conclusion

This project used Online Retail transaction data to segment customers using RFM behavior. The final model creates three groups:

1. **High-Value Customers**
2. **Regular / Growth Potential Customers**
3. **At-Risk / Low-Engagement Customers**

The final recommendation is to use different marketing actions for each segment instead of sending one generic campaign to all customers.

This analysis supports a clearer and more targeted marketing strategy.

# AI assistance disclosure

AI tools were used as support to organize the notebook, improve explanations, and help structure the code. The team reviewed the notebook and must be able to explain every line of code, every chart, and every business recommendation in the presentation.

# Reproducibility checklist

- `RANDOM_STATE = 42` is set.
- The notebook does not use absolute local paths.
- The dataset loads from the same folder as the notebook.
- The notebook creates CSV outputs, chart images, and an HTML dashboard.
- The notebook is designed to run from top to bottom with **Restart & Run All**.